## This notebook deals with PySpark code of dealing with Aggregations such as Group By, Having, Collect Set, Collect List and advanced aggregations such as Window functions  

In [0]:
df=spark.read.table("samples.bakehouse.sales_transactions")
display(df)

In [0]:
df.count()

we dont have df.last() because PySpark df are unordered by default so we have no guarenteed last row 

In [0]:
# df.first()
df.head()

In [0]:
from pyspark.sql.functions import min,max,col
df2=df.select(min(col("quantity")), max(col("quantity")))
#df2=df.select(min(("quantity")), max(("quantity")))
df2.display()

In [0]:
from  pyspark.sql.functions import sum, avg, col
# df2=df.select(sum("quantity"), avg("quantity"))
df2=df.select(sum(col("quantity")), avg(col("quantity")))
display(df2)


In [0]:
%sql
select count(*),min(quantity),max(quantity),sum(quantity),avg(quantity)
from samples.bakehouse.sales_transactions

for groupBy to work we need to pair it up with an aggregate function

In [0]:
df2=df.groupBy("paymentMethod","product").count()
df2.display()

In [0]:
%sql
select 
    paymentMethod,
    product,
    count(*)
from samples.bakehouse.sales_transactions
group by paymentMethod,product


for doing operation of Having in Group By we use .filter or .where clauses and define the col where we want the filter and define the filtering condition

In [0]:
from pyspark.sql.functions import col,desc
df2=df.groupBy("paymentMethod","product").agg(sum(col("quantity")).alias("count")).filter(col("count")>1200).orderBy(desc("count"))
df2.display()

In [0]:
%sql
select 
    paymentMethod,
    product,
    sum(quantity) as count 
from samples.bakehouse.sales_transactions
group by paymentMethod,product
having count > 1200
order by count desc

### collect_list()
- Collects all values from a column into an array, including duplicates.
- Does not guarantee the order of elements.

### collect_set()
- Collects unique values from a column into an array, removing duplicates.
- Does not guarantee the order of elements.

In [0]:
from pyspark.sql.functions import collect_set, collect_list
df2=df.groupBy("paymentMethod").agg(collect_set("product"), collect_list("product"))
display(df2)

In [0]:
%sql
select 
    paymentMethod,
    collect_set(product),
    collect_list(product)
from samples.bakehouse.sales_transactions
group by paymentMethod

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,desc

window_spec=Window.partitionBy("product").orderBy(desc("quantity"))
df2=df.withColumn("row_number_over_product", row_number().over(window_spec))
df3=df2.select("product","customerID","quantity","row_number_over_product")
df3.display()

In [0]:
df4=df3.filter(df3["row_number_over_product"]<5)
df4.display()

In [0]:
#row_number=1 means that max quantity purchased since quantity is being orderedBy in desc order so max quantity is at row_number=1 and corresponding to that we are fetching the custokmerIDs as well
df5=df3.filter("row_number_over_product=1")
df5.display()

In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,rank,dense_rank

window_spec=Window.partitionBy("product").orderBy(desc("quantity"))
df6=df.withColumn("row_number_over_product", row_number().over(window_spec)).\
    withColumn("rank_over_product", rank().over(window_spec)).\
        withColumn("dense_rank_over_product", dense_rank().over(window_spec))
df6.display()

In [0]:
%sql
select 
    product,
    customerID,
    quantity,
    row_number() over (partition by product order by quantity desc) as row_number_over_product,
    rank() over (partition by product order by quantity desc) as rank_over_product,
    dense_rank() over (partition by product order by quantity desc) as dense_rank_over_product
from samples.bakehouse.sales_transactions